# Java 코드 추적 문제은행

상속·필드·오버라이딩·문자열을 묶었다. 맞혔던 유형도 변형에서 흔들리지 않도록 푼다.

답안 칸에 먼저 답을 쓰고, 바로 아래의 정답·해설을 펼쳐 확인하세요. #는 오답 표시로 사용하세요.

## 1회 유형 · 심화

기존 문제의 답안 기록은 백업에 보존하고, 이 새 문제은행의 답안 칸은 비웠습니다.

## 이 은행에서 반복해서 묻는 것

| 함정 | 어디서 나오나 |
|---|---|
| 오버로딩 = 컴파일 시점 / 오버라이딩 = 실행 시점 | Q1, Q5 |
| 메서드는 객체 따라, 필드는 타입 따라 | Q2, Q19 |
| `+` 좌결합과 문자열 전환 지점 | Q3 |
| 초기화 순서 (static → 인스턴스 → 생성자) | Q4, Q16 |
| `==` vs `equals` (String, Integer) | Q6, Q8 |
| finally가 return을 덮어씀 | Q7, Q11 |
| `remove(int)` vs `remove(Object)` | Q10 |
| 삼항 연산자 타입 승격 | Q12 |

---

## Q1. 오버로딩 결정 시점 vs 오버라이딩 (26년1회 11번 유형)

```java
class A {
    String f(Object x) { return "1"; }
    String g() { return f("hello"); }
}

class B extends A {
    String f(Object x) { return "2"; }
    String f(String x) { return "3"; }
}

public class Test {
    public static void main(String[] args) {
        A a = new B();
        System.out.println(a.g());
    }
}
```

**출력 결과를 쓰시오.**

<details>
<summary><b>👉 정답 · 해설 보기</b></summary>

```
2
```

이 시험에서 가장 자주 틀리는 문제 유형이다. 두 단계를 나눠서 봐야 한다.

**1단계 — 어느 메서드를 부를지(오버로딩)는 컴파일 때 결정된다.**
`g()` 는 클래스 A 안에 있고, A에는 `f(Object)` 밖에 없다.
그래서 `f("hello")` 는 **`f(Object)` 호출로 고정**된다. B의 `f(String)` 은 A가 모르므로 후보에 없다.

**2단계 — 그 메서드의 실제 구현(오버라이딩)은 실행 때 결정된다.**
객체가 B이므로 B가 재정의한 `f(Object)` 가 실행된다 → **"2"**

"가장 구체적인 f(String)이 불리겠지"라고 생각하면 3이라고 쓰게 되는데 오답이다.

</details>

## Q2. 필드와 메서드의 접근 규칙

```java
class A {
    int x = 1;
    int get() { return x; }
}

class B extends A {
    int x = 2;
    int get() { return x; }
}

public class Test {
    public static void main(String[] args) {
        A obj = new B();
        System.out.println(obj.x + " " + obj.get());
    }
}
```

**출력 결과를 쓰시오.**

<details>
<summary><b>👉 정답 · 해설 보기</b></summary>

```
1 2
```

**메서드는 객체 따라, 필드는 참조변수 타입 따라.** 이 한 줄이면 끝나는 문제다.

- `obj.x` : 참조변수 타입이 A이므로 **A의 x = 1**
- `obj.get()` : 실제 객체가 B이므로 **B의 get() → B의 x = 2**

필드는 오버라이딩되지 않고 **가려지기(hiding)** 만 한다.

</details>

## Q3. char 산술과 + 연산 순서 (26년1회 19번 유형)

```java
public class Main {
    int func(int a, int b) { return a + b; }
    int func(char a, char b) { return b - a; }
    char func(char a) { return a; }

    public static void main(String[] args) {
        Main m = new Main();
        int r1 = m.func(4, 5);
        int r2 = m.func('A', 'C');
        char r3 = m.func('3');
        System.out.println(r1 + r2 + "2" + r3);
    }
}
```

**출력 결과를 쓰시오.**

<details>
<summary><b>👉 정답 · 해설 보기</b></summary>

```
1123
```

`+` 는 **왼쪽부터** 계산하고, 문자열을 만나는 순간부터 뒤는 전부 연결이 된다.

- `r1` = 4+5 = **9**
- `r2` = `'C'-'A'` = 67-65 = **2**  (char끼리 빼면 int가 된다)
- `r3` = 문자 **'3'**
- `r1 + r2` = 11 → 아직 숫자 덧셈
- `11 + "2"` = **"112"** → 여기서부터 문자열
- `"112" + '3'` = **"1123"**

만약 `"" + r1 + r2 + "2" + r3` 였다면 처음부터 문자열이라 **"9223"** 이 된다.

</details>

## Q4. static 블록 · 인스턴스 블록 · 생성자 순서

```java
class A {
    static { System.out.print("1"); }
    { System.out.print("2"); }
    A() { System.out.print("3"); }
}

public class Test {
    public static void main(String[] args) {
        new A();
        new A();
        System.out.println();
    }
}
```

**출력 결과를 쓰시오.**

<details>
<summary><b>👉 정답 · 해설 보기</b></summary>

```
12323
```

실행 순서는 **static 블록 → 인스턴스 블록 → 생성자** 다.

- static 블록은 **클래스가 처음 로딩될 때 딱 한 번** → "1"
- 객체를 만들 때마다 인스턴스 블록 → 생성자 → "23"
- 두 번 만들었으므로 "23" 이 두 번

합쳐서 **12323**

</details>

## Q5. static 메서드 숨김(hiding)

```java
class A {
    static String f() { return "A"; }
    String g() { return "a"; }
}

class B extends A {
    static String f() { return "B"; }
    String g() { return "b"; }
}

public class Test {
    public static void main(String[] args) {
        A obj = new B();
        System.out.println(A.f() + " " + B.f());
    }
}
```

**출력 결과를 쓰시오.**

<details>
<summary><b>👉 정답 · 해설 보기</b></summary>

```
A B
```

static 메서드는 **오버라이딩이 아니라 숨김(hiding)** 이다. 객체가 아니라 **클래스 이름**으로 결정된다.
`A.f()` 는 A의 것, `B.f()` 는 B의 것을 부른다.

인스턴스 메서드 `g()` 였다면 실제 객체 B를 따라갔겠지만, static은 그렇지 않다.

</details>

## Q6. String 상수 풀과 비교

```java
public class Test {
    public static void main(String[] args) {
        String a = "abc";
        String b = "abc";
        String c = new String("abc");
        System.out.println((a == b) + " " + (a == c) + " " + a.equals(c));
    }
}
```

**출력 결과를 쓰시오.**

<details>
<summary><b>👉 정답 · 해설 보기</b></summary>

```
true false true
```

- `a == b` : 같은 리터럴은 **상수 풀의 같은 객체**를 가리킨다 → **true**
- `a == c` : `new String(...)` 은 **힙에 새 객체**를 만든다 → **false**
- `a.equals(c)` : 값 비교 → **true**

**문자열 값 비교는 언제나 `equals()`.** `==` 는 주소 비교다.

</details>

## Q7. finally 안의 return

```java
public class Test {
    static int f() {
        try {
            return 1;
        } finally {
            System.out.print("F ");
            return 2;
        }
    }
    public static void main(String[] args) {
        System.out.println(f());
    }
}
```

**출력 결과를 쓰시오.**

<details>
<summary><b>👉 정답 · 해설 보기</b></summary>

```
F 2
```

`try` 의 `return 1` 이 준비되더라도 **finally는 반드시 실행**되고,
finally 안에 return이 있으면 **그 값이 최종 반환값을 덮어쓴다** → **2**

finally에 return을 쓰지 말라는 이유가 이것이다. try의 결과가 조용히 사라진다.

</details>

## Q8. Integer 객체 비교

```java
public class Test {
    public static void main(String[] args) {
        Integer a = 127, b = 127;
        Integer c = 128, d = 128;
        System.out.println((a == b) + " " + (c == d) + " " + c.equals(d));
    }
}
```

**출력 결과를 쓰시오.**

<details>
<summary><b>👉 정답 · 해설 보기</b></summary>

```
true false true
```

자바는 **-128 ~ 127 범위의 Integer를 캐시**해 같은 객체를 재사용한다.

- 127은 캐시 범위 안 → 같은 객체 → **true**
- 128은 범위 밖 → 각각 새 객체 → **false**
- `equals` 는 값 비교라 언제나 **true**

래퍼 클래스도 `==` 가 아니라 `equals` 로 비교해야 한다는 근거다.

</details>

## Q9. 배열 기본값과 length

```java
public class Test {
    public static void main(String[] args) {
        int[] a = new int[3];
        String[] b = new String[2];
        System.out.println(a[0] + " " + b[0] + " " + a.length + " " + "Hello".length());
    }
}
```

**출력 결과를 쓰시오.**

<details>
<summary><b>👉 정답 · 해설 보기</b></summary>

```
0 null 3 5
```

- `int[]` 의 기본값은 **0**, 참조형 배열의 기본값은 **null**
- 배열은 `length` (**괄호 없는 필드**), 문자열은 `length()` (**괄호 있는 메서드**)

이 둘을 바꿔 쓰면 컴파일 오류다. 서술형에서도 자주 묻는다.

</details>

## Q10. ArrayList remove(int) 와 remove(Object)

```java
import java.util.ArrayList;

public class Test {
    public static void main(String[] args) {
        ArrayList<Integer> a = new ArrayList<>();
        a.add(10); a.add(20); a.add(30); a.add(40);
        a.remove(1);
        a.remove(Integer.valueOf(30));
        System.out.println(a + " " + a.size());
    }
}
```

**출력 결과를 쓰시오.**

<details>
<summary><b>👉 정답 · 해설 보기</b></summary>

```
[10, 40] 2
```

- `remove(1)` : 인자가 **int** 라 **인덱스 1**(값 20)을 지운다 → [10, 30, 40]
- `remove(Integer.valueOf(30))` : 인자가 **객체**라 **값 30**을 지운다 → [10, 40]

같은 이름인데 인자 타입에 따라 뜻이 완전히 달라진다. 출력은 `[10, 40]` 형태 그대로 쓴다.

</details>

## Q11. 예외 처리 흐름

```java
public class Test {
    static int f(int[] a, int i) {
        try {
            return a[i];
        } catch (ArrayIndexOutOfBoundsException e) {
            System.out.print("B ");
            return -1;
        } finally {
            System.out.print("C ");
        }
    }
    public static void main(String[] args) {
        int[] a = {1, 2, 3};
        System.out.println(f(a, 5) + f(a, 2) + 1);
    }
}
```

**출력 결과를 쓰시오.**

<details>
<summary><b>👉 정답 · 해설 보기</b></summary>

```
B C C 3
```

- `f(a,5)` : 인덱스 초과 → catch 실행("B ") → finally("C ") → **-1** 반환
- `f(a,2)` : 정상 → finally만 실행("C ") → **3** 반환
- `-1 + 3 + 1` = **3**

출력 순서에 주의한다. 두 호출이 **모두 println의 인자로 먼저 계산**되므로
"B C C " 가 먼저 다 찍히고 마지막에 3이 나온다.

</details>

## Q12. 삼항 연산자의 타입 승격

```java
public class Test {
    public static void main(String[] args) {
        int x = 5;
        System.out.println(x > 3 ? 1 : 2.0);
    }
}
```

**출력 결과를 쓰시오.**

<details>
<summary><b>👉 정답 · 해설 보기</b></summary>

```
1.0
```

삼항 연산자는 **두 갈래의 타입을 하나로 맞춘다.** `int` 와 `double` 이 섞이면 **double로 승격**된다.
그래서 1이 아니라 **1.0** 이 출력된다.

조건이 참이라 1만 보고 "1"이라고 쓰면 오답이다.

</details>

## Q13. 증감 연산자 복합

```java
public class Test {
    public static void main(String[] args) {
        int i = 5;
        int a = i++;
        int b = ++i;
        int c = i--;
        System.out.println(a + " " + b + " " + c + " " + i);
    }
}
```

**출력 결과를 쓰시오.**

<details>
<summary><b>👉 정답 · 해설 보기</b></summary>

```
5 7 7 6
```

| 식 | 쓰이는 값 | 실행 후 i |
|---|---|---|
| `i++` | 5 | 6 |
| `++i` | 7 | 7 |
| `i--` | 7 | 6 |

**후위는 쓰고 나서 변하고, 전위는 변하고 나서 쓴다.**

</details>

## Q14. HashMap 중복 키와 없는 키

```java
import java.util.HashMap;

public class Test {
    public static void main(String[] args) {
        HashMap<String, Integer> m = new HashMap<>();
        m.put("a", 1);
        m.put("b", 2);
        m.put("a", 3);
        System.out.println(m.size() + " " + m.get("a") + " " + m.get("z"));
    }
}
```

**출력 결과를 쓰시오.**

<details>
<summary><b>👉 정답 · 해설 보기</b></summary>

```
2 3 null
```

- 같은 키에 다시 `put` 하면 **값만 덮어쓴다** → 크기는 여전히 **2**
- `m.get("a")` = **3**
- 없는 키를 `get` 하면 예외가 아니라 **null** 을 반환한다

> 없는 키의 결과는 답안에 **null** 이라고 쓴다. 0이나 공백이 아니다.

</details>

## Q15. 문자열 메서드 조합

```java
public class Test {
    public static void main(String[] args) {
        String s = "Information";
        System.out.println(s.length() + " " + s.substring(0, 4) + " "
            + s.substring(4, 7) + " " + s.indexOf("m") + " " + s.replace("o", "0"));
    }
}
```

**출력 결과를 쓰시오.**

<details>
<summary><b>👉 정답 · 해설 보기</b></summary>

```
11 Info rma 5 Inf0rmati0n
```

- `length()` = **11**
- `substring(0,4)` = "Info" (**끝 인덱스는 포함하지 않는다**)
- `substring(4,7)` = "rma"
- `indexOf("m")` : I(0) n(1) f(2) o(3) r(4) m(5) → **5**
- `replace("o","0")` : 소문자 o만 두 군데 바뀐다 → "Inf0rmati0n"

</details>

## Q16. this() 와 super() 생성자 체이닝

```java
class A {
    A() { System.out.print("A"); }
    A(int x) { this(); System.out.print(x); }
}

class B extends A {
    B() { super(100); System.out.print("B"); }
}

public class Test {
    public static void main(String[] args) {
        new B();
        System.out.println();
    }
}
```

**출력 결과를 쓰시오.**

<details>
<summary><b>👉 정답 · 해설 보기</b></summary>

```
A100B
```

`new B()` → `super(100)` → A(int)가 먼저 `this()` 로 A()를 부른다.

1. `A()` 실행 → "A"
2. `A(int)` 의 남은 부분 → "100"
3. `B()` 의 남은 부분 → "B"

**"A100B"**. 생성자는 항상 **부모가 먼저, 안쪽이 먼저** 끝난다.

</details>

## Q17. 인터페이스와 다형성 배열

```java
interface Shape { int area(); }

class Rect implements Shape {
    int w, h;
    Rect(int w, int h) { this.w = w; this.h = h; }
    public int area() { return w * h; }
}

class Square extends Rect {
    Square(int s) { super(s, s); }
}

public class Test {
    public static void main(String[] args) {
        Shape[] arr = { new Rect(3, 4), new Rect(2, 8), new Square(5) };
        for (Shape s : arr) System.out.print(s.area() + " ");
        System.out.println();
    }
}
```

**출력 결과를 쓰시오.**

<details>
<summary><b>👉 정답 · 해설 보기</b></summary>

```
12 16 25
```

배열의 선언 타입은 `Shape` 지만, `area()` 는 **실제 객체의 것**이 불린다.
Square는 Rect의 생성자를 통해 w=h=5가 되므로 25.

3×4=12, 2×8=16, 5×5=25

</details>

## Q18. 향상된 for문 종합

```java
public class Test {
    public static void main(String[] args) {
        int[] a = {3, 8, 1, 9, 4, 7};
        int s = 0, mx = a[0], cnt = 0;
        for (int x : a) {
            if (x % 2 == 1) s += x;
            if (x > mx) mx = x;
            if (x > 4) cnt++;
        }
        System.out.println(s + " " + mx + " " + cnt);
    }
}
```

**출력 결과를 쓰시오.**

<details>
<summary><b>👉 정답 · 해설 보기</b></summary>

```
20 9 3
```

- 홀수 합 : 3+1+9+7 = **20**
- 최댓값 : **9**
- 4보다 큰 것 : 8, 9, 7 → **3개**

향상된 for문은 **읽기 전용**이다. `x` 를 바꿔도 배열 원본은 변하지 않는다.

</details>

## Q19. 추상 클래스와 필드 상속

```java
abstract class A {
    int x = 10;
    abstract int f();
    int g() { return x + f(); }
}

class B extends A {
    int y = 20;
    int f() { return y; }
}

public class Test {
    public static void main(String[] args) {
        B b = new B();
        System.out.println(b.x + " " + b.y + " " + b.g());
    }
}
```

**출력 결과를 쓰시오.**

<details>
<summary><b>👉 정답 · 해설 보기</b></summary>

```
10 20 30
```

추상 클래스도 **필드와 일반 메서드를 가질 수 있다.**
`g()` 는 A에 있지만 그 안의 `f()` 는 B의 구현을 부른다 → 10 + 20 = **30**

추상 메서드만 있는 게 인터페이스, 필드·구현을 섞을 수 있는 게 추상 클래스다.

</details>

## Q20. 자릿수 합과 역순 만들기

```java
public class Test {
    public static void main(String[] args) {
        int n = 9074, s = 0, rev = 0;
        while (n > 0) {
            int d = n % 10;
            s += d;
            rev = rev * 10 + d;
            n /= 10;
        }
        System.out.println(s + " " + rev);
    }
}
```

**출력 결과를 쓰시오.**

<details>
<summary><b>👉 정답 · 해설 보기</b></summary>

```
20 4709
```

9074를 뒤에서부터 한 자리씩 떼어낸다 : 4, 7, 0, 9

- 합 : 4+7+0+9 = **20**
- 역순 : 4 → 47 → 470 → 4709 → **4709**

`rev * 10 + d` 는 자릿수를 밀어 올리는 표준 관용구다.

</details>

## 2회 유형 · 기본/중간

기존 문제의 답안 기록은 백업에 보존하고, 이 새 문제은행의 답안 칸은 비웠습니다.

## Q1. private 필드와 오버라이딩 (26년2회 18번 유형)

```java
class A {
    int a;
    private int b;
    protected int c;

    A(int aa, int bb, int cc) { a = aa; b = bb; c = cc; }

    int hap() { return a + b + c; }
}

class B extends A {
    B(int aa, int bb, int cc) { super(aa, bb, cc); }

    @Override
    public int hap() { return a * c; }
}

public class Main {
    public static void main(String[] args) {
        A x = new A(1, 5, 3);
        B y = new B(10, 30, 50);
        System.out.print(x.hap() + y.hap());
    }
}
```

**출력 결과를 쓰시오.**

<details>
<summary><b>👉 정답 · 해설 보기</b></summary>

```
509
```

두 가지를 동시에 묻는다.

**1. `private int b` 는 자식이 못 본다.** B의 `hap()` 이 `a * c` 만 쓰는 이유가 그것이다.
객체 안에 b=30이 저장은 되어 있지만 B의 코드에서는 접근할 수 없다.

**2. 어느 `hap()` 이 불리는가.**
- `x.hap()` : x는 A 객체 → A의 hap → 1 + 5 + 3 = **9**
- `y.hap()` : y는 B 객체 → **B가 재정의한 hap** → 10 × 50 = **500**

9 + 500 = **509**

y의 b=30은 어디에도 안 쓰인다. 이 미끼에 걸려 10+30+50을 더하면 틀린다.

</details>

## Q2. super 호출과 출력 순서 (26년2회 9번 유형)

```java
class A {
    int a;
    A(int a) { this.a = a; }
    void print() { System.out.print(a + "a"); }
}

class B extends A {
    int b;
    B(int a, int b) { super(a); this.b = b; }

    @Override
    void print() {
        super.print();
        System.out.print(b + "b");
    }
}

public class Main {
    public static void main(String[] args) {
        new B(10, 20).print();
    }
}
```

**출력 결과를 쓰시오.**

<details>
<summary><b>👉 정답 · 해설 보기</b></summary>

```
10a20b
```

`super.print()` 는 **부모의 구현을 먼저 실행**하고 돌아온다. 그래서 부모 출력이 앞에 온다.

- A의 print : `a + "a"` → a는 int(10), `+` 뒤에 문자열이 오므로 **"10a"**
- 돌아와서 B의 나머지 : `b + "b"` → **"20b"**

합쳐서 **10a20b**

`10 + "a"` 가 11이 아니라 "10a"가 되는 건, `+` 한쪽이 문자열이면 **연결**이 되기 때문이다.

</details>

## Q3. 재귀 — 음수 인자 (26년2회 4번 유형)

```java
public class Main {
    static int compute(int num) {
        if (num <= 1) return num;
        return compute(num - 3) + compute(num - 1);
    }

    public static void main(String[] args) {
        System.out.print(compute(5) + " " + compute(7));
    }
}
```

**출력 결과를 쓰시오.**

<details>
<summary><b>👉 정답 · 해설 보기</b></summary>

```
1 2
```

`num - 3` 때문에 인자가 **음수까지 내려가고**, 종료 조건이 `return num` 이라 **음수가 그대로 반환**된다.

compute(5) = compute(2) + compute(4)
- compute(2) = compute(-1) + compute(1) = **-1** + 1 = 0
- compute(4) = compute(1) + compute(3) = 1 + 0 = 1
- → **1**

compute(7) = compute(4) + compute(6) = 1 + (compute(3) + compute(5)) = 1 + (0 + 1) = **2**

재귀는 반드시 **호출 트리를 그려서** 아래에서 위로 값을 채워 올라간다. 암산하면 반드시 틀린다.

</details>

## Q4. 접근 제어자 범위

```java
class A {
    public    int w = 1;
    protected int x = 2;
              int y = 3;   // default (package-private)
    private   int z = 4;

    int all() { return w + x + y + z; }
}

class B extends A {
    int sub() { return w + x + y; }
}

public class Main {
    public static void main(String[] args) {
        B b = new B();
        System.out.print(b.all() + " " + b.sub());
    }
}
```

**출력 결과를 쓰시오.**

<details>
<summary><b>👉 정답 · 해설 보기</b></summary>

```
10 6
```

- `b.all()` : A에서 정의된 메서드라 **자기 클래스의 private에도 접근 가능** → 1+2+3+4 = **10**
- `b.sub()` : B에서 정의됐으므로 `z`(private)에 접근 불가 → 1+2+3 = **6**

| 제어자 | 같은 클래스 | 같은 패키지 | 자식 클래스 | 전체 |
|---|---|---|---|---|
| `private` | O | X | X | X |
| (default) | O | O | 같은 패키지만 | X |
| `protected` | O | O | O | X |
| `public` | O | O | O | O |

**private은 "클래스 기준"이지 "객체 기준"이 아니다.** A의 메서드는 A의 private을 늘 볼 수 있다.

</details>

## Q5. 문자열 연결과 산술의 경계

```java
public class Main {
    public static void main(String[] args) {
        int a = 10, b = 20;
        System.out.println(a + b + "c");
        System.out.println("c" + a + b);
        System.out.println(a + b + "c" + a + b);
        System.out.println('A' + 1 + "B");
    }
}
```

**출력 결과를 쓰시오.**

<details>
<summary><b>👉 정답 · 해설 보기</b></summary>

```
30c
c1020
30c1020
66B
```

`+` 는 **왼쪽부터** 계산하고, **문자열을 만나는 순간부터** 뒤는 전부 연결이 된다.

- `a + b + "c"` : 10+20=30 → "30c"
- `"c" + a + b` : 처음부터 문자열 → "c10" → **"c1020"**
- `a + b + "c" + a + b` : 30 → "30c" → "30c10" → **"30c1020"**
- `'A' + 1 + "B"` : char + int는 **int 산술** → 65+1=66 → **"66B"**

마지막 줄에서 "AB"나 "B1B"라고 쓰면 오답이다. `'A' + 1` 은 문자가 아니라 **66**이다.

</details>